In [74]:
##### viewing tracking algorithm results
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time

check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/Project/'
data=xr.open_dataset(dir+'cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
parcel=parcel.isel(time=np.arange(0, len(parcel['time']), int(times[1]))) 
whereCL=xr.open_dataset(dir+'whereCL_4_0622.nc') #***

out=xr.open_dataset(dir+'job_array/parcel_tracking4tundra-7_062217.nc')['out_arr'].values;out=out.astype(object);out[:, [0,1,2,4,5]] = out[:, [0,1,2,4,5]].astype(int) #***
save=xr.open_dataset(dir+'job_array/parcel_tracking4tundra-7_062217.nc')['save_arr'].values;save=save.astype(object);save[:, [0,1,2,4,5]] = save[:, [0,1,2,4,5]].astype(int) #***

out_nz=out[~np.all(out == 0, axis=1)];print('list of first 10 SBZ parcels'); print(out_nz[:15])
save_nz=save[~np.all(save == 0, axis=1)];save_nz=save_nz[np.where(np.unique(save_nz[1:-1,0]))];print('list of first 10 ignored parcels');print(save_nz[:5])

###############################################################################
#remove duplicates
lst=[]
unique_values, counts = np.unique(out_nz[:,0], return_counts=True); duplicates = unique_values[counts > 1]
for elem in duplicates:
    idx = np.where(out_nz[:,0] == elem)[0] 
    extras=idx[np.where(out_nz[idx,5]!=np.min(out_nz[idx,5]))]
    lst.extend([x for x in extras])
mask=np.ones(len(out_nz), dtype=bool); mask[lst] = False
out_nz=out_nz[mask]; 
###############################################################################
print(f'there are a total of {len(out_nz)} SBZ parcels and {len(save_nz)} forgotten parcels')

#################################################################################################################################
if check==True:
    list=[]
    print('checking if all found SBZ parcels originate from CL')
    for ind in range(0,out_nz.shape[0]):
        x=parcel['x'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000;xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1 
        y=parcel['y'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000;yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1 
        z=parcel['z'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1;
        maxconv_x=whereCL['maxconv_x'].isel(time=out_nz[ind,4],y=which_y,z=which_z).values
        list.append(any(np.isin(maxconv_x, np.arange(which_x-2,which_x+3))))
    print(list[:20])
    list=[]
    print('checking if all found SBZ parcels hit LFC')
    for ind in range(0,out_nz.shape[0]):
        xloc=parcel['x'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000; yloc=parcel['y'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
        xf=data['xf'].values; which_x=np.searchsorted(xf,xloc)-1; yf=data['yf'].values; which_y=np.searchsorted(yf,yloc)-1; 
        z=parcel['z'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000 
        lfc=data['lfc'].isel(time=out_nz[ind,5],xh=which_x,yh=which_y).values/1000
        list.append(z>=lfc)
    print(list[:20])
#################################################################################################################################

out_min=np.round(np.min(out_nz[:,3]),3);out_max=np.round(np.max(out_nz[:,3]),3); print(f"CL zlev range is {out_min}:{out_max:.3f} km")
out_mean=np.round(np.mean(out_nz[:,3]),3); print(f'mean CL zlev is {out_mean} km')
placeholder=out_nz.copy(); run=True

list of first 10 SBZ parcels
[[1 318 61 0.29015762329101563 33 38]
 [4 443 12 0.15114982604980468 34 39]
 [4 433 12 0.0640534439086914 8 13]
 [6 411 83 0.06429761505126953 72 78]
 [12 302 69 0.20426243591308593 567 571]
 [10 462 47 0.09030453491210938 276 284]
 [17 342 89 0.19663821411132812 294 298]
 [27 443 1 0.01788448905944824 568 574]
 [38 415 67 0.3010951843261719 303 307]
 [34 295 84 0.029517009735107423 22 27]
 [42 358 40 0.09808511352539062 34 38]
 [48 330 3 0.04794386672973633 37 44]
 [63 426 52 0.10616990661621094 276 282]
 [79 399 10 0.34620989990234374 278 281]
 [84 372 37 0.13176063537597657 309 313]]
list of first 10 ignored parcels
[[0 166 68 1.2079783935546875 571 572]
 [1 498 57 1.53133154296875 555 556]
 [4 375 78 7.54065185546875 351 353]
 [5 51 96 1.124783203125 567 568]
 [9 281 81 0.812075927734375 474 475]]
there are a total of 1891 SBZ parcels and 4177 forgotten parcels
CL zlev range is 0.003:0.733 km
mean CL zlev is 0.2 km


In [75]:
#search for deep convective parcels (comment if want to show all tracked parcels)        
##############################################################
def threshold(zthresh):
    out_nz=placeholder.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_nz)): 
        #searchs if next most local max goes above zthresh
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_nz[ind,5],out_nz[ind,5]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
        above=parcel['z'].isel(xh=out_nz[ind,0],time=aboverange).values/1000
    
        dt=1
        #takes dabove/dt
        f=above
        ddx = (
                f[1:  ]
                -
                f[0:-1]
            ) / (
            2 * dt
        )
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        
        if np.any(above[local_maxes[0]]>zthresh):
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)
    
    out_nz=out_nz[deep_out_ind,:]
    print(f'> {zthresh} km. {len(out_nz)} leftover parcels')
    return out_nz, extendrange
    # print(out_nz)
##############################################################

#getting info for variables to plot (better info)
#includes w, qv, th, and budget variables
ds=data.copy()
variables = ds.variables
long_names = {var: {
        'long_name': ds[var].attrs.get('long_name', ''),
        'units': ds[var].attrs.get('units', '')
    } for var in variables}


w_vars = {var: long_name for var, long_name in long_names.items() if var.startswith('wb') or var in ['w']}
ptb_vars = {var: long_name for var, long_name in long_names.items() if var.startswith('ptb') or var in ['th']}
qvb_vars = {var: long_name for var, long_name in long_names.items() if var.startswith('qvb') or var in ['qv']}
vars = {}
vars.update(w_vars)
vars.update(ptb_vars)
vars.update(qvb_vars)


var_names=list(vars.keys())
long_names = [info['long_name'] for info in vars.values()]
units = [info['units'] for info in vars.values()]
def convert_to_latex(unit):
    parts = unit.split('/')  # Split the unit into parts
    latex_unit = parts[0]  # Start with the first part
    for part in parts[1:]:  # Iterate over the remaining parts
        latex_unit += r"\ " + part + r"^{-1}"  # Add the part with a '^{-1}' exponent
    return r"$" + latex_unit + r"$"
latex_units = [convert_to_latex(unit) for unit in units]


#adding qv prime
var_names.append(f"qv'")
long_names.append('perturbation water vapor mixing ratio')
latex_units.append(r'$kg\ kg^{-1}')

# #adding qv*w (vertical moisture transport)  
# var_names.append('qvw')
# long_names.append('vertical moisture transport')
# latex_units.append(r'$kg\ kg^{-1}\ m\ s^{-1}$')

# #adding convergence of qv*w (moisture flux convergence) #issues calculating
# var_names.append('qvwconv')
# long_names.append('vertical moisture flux convergence')
# latex_units.append(r'$kg\ kg^{-1}\ s^{-1}$')

In [ ]:
############################################################################################################################################################

In [10]:
#job array setup
num_jobs=15 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
if job_id==0: job_id=1

num_parcels=len(var_names) #total num of variables
job_range = num_parcels//num_jobs #number of parcels per job 
if job_id==num_jobs: end_job=num_parcels
start_job = (job_id - 1) * job_range
end_job = start_job + job_range
if job_id==num_jobs: end_job=num_parcels-1

In [ ]:
#Plotting data 2x2x2 box (zmean) around parcel from CL to LFC *Mean of convective parcels* ADDED JOB ARRAY
#
# (1) runs through all parcel 
# (2) time slice from CL-LFC is found
# (3) time slice is used to find the parcels x,y, and z grid locations (interpolated to 7+-1 times)
# (4) for each time, a 2x2 km box around parcel is added to empty array for that time
# (5) creates figures
#
start_time = time.time()
if run==True:
    out_nz2=out_nz.copy()
    zthresh=5
    [out_nz,extendrange]=threshold(zthresh)
    run=False

def data_save(var,arr,pdata_mean,savename):
    folder_path='/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/trajectories/'
    arr[var+'_pdata_mean'] = pdata_mean
    np.save(folder_path+savename+'_array.npy', arr)
    return arr

timing=[]
for ind in range(7+1):
    timing.append([])
def cllfc_mean(var,unit,long_name):
    global out_nz, timing #*timing
    contours=[] #colorbar fix 
   
    def read_var(slice, x, y, z, var):
        global timing #*timing1
        timing_one=time.time() #*timing1

        timing_seven=time.time() #*timing7
        data['time']=np.arange(len(data['time']))
        coord_map = {
            'xh': 'xf',
            'yh': 'yf',
            'zh': 'zf'
        }
        
        var_coords = data[var].coords
        coords=['xh', 'yh', 'zh'] 
        for ind, coord in enumerate(coords):
            if coord not in var_coords:
                coords[ind] = coord_map[coord]
        indices = {}
        indices[coords[0]] = x
        indices[coords[1]] = y
        indices[coords[2]] = z
        # output=data[var].isel(**{coord: indices[coord] for coord in coords}) #old 
        ############################
        output=data[var].isel(**{coord: indices[coord] for coord in coords[0:2]}) #splitting horizontal and vertical
        output=output.isel(**{coord: indices[coord] for coord in [coords[2]]}) #splitting horizontal and vertical
        timing_two=time.time() #*timing7
        timing[7].append(timing_two-timing_one) #*timing7
    
        #interpolation
        timing_one=time.time() #*timing6
        t1=slice[0];t2=slice[-1]
        even=(t2-t1)/n
        new_t=np.linspace(t1, t2, n+1)
        new_t=np.insert(new_t,0,new_t[0]-even)
        new_t=np.append(new_t,new_t[-1]+even)
        output=output.interp(time=new_t,method='linear') 
        
        # ##### TESTING
        # output = output.isel(time=np.arange(int(np.floor(new_t[0])), int(np.ceil(new_t[-1]))+1))   
        # output = output.interp(time=new_t, method='linear')
        # ##### TESTING
        
        timing_two=time.time() #*timing6
        timing[6].append(timing_two-timing_one) #*timing6

        timing_two=time.time() #*timing1
        timing[1].append(timing_two-timing_one) #*timing1
        
        return output

    def parcel_read(slice,xh,var):
        global timing #*timing2
        timing_one=time.time() #*timing2
        
        parcel['time']=np.arange(len(parcel['time']))
        output=parcel[var].isel(xh=row[0])
        #interpolation
        t1=slice[0];t2=slice[-1]
        even=(t2-t1)/n
        new_t=np.linspace(t1, t2, n+1)
        new_t=np.insert(new_t,0,new_t[0]-even)
        new_t=np.append(new_t,new_t[-1]+even)
        output=output.interp(time=new_t,method='linear').values

        timing_two=time.time() #*timing2
        timing[2].append(timing_two-timing_one) #*timing2
        return output
    
    zlev=2
    kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #finds how many x grids is 1 km
    ns=2*kms
    n=int(30/times[1]) #number of times to split CL-LFC trajectory into
    
    pdata_mean=np.zeros(((n+2)+1,2*ns+1,2*ns+1))
    for prnt, row in enumerate(out_nz[start_job:end_job+1]): #for job_array
        if np.mod(prnt,25)==0: print(f'{prnt}/{len(out_nz)} parcels done') #TESTING
            
        slice= np.arange(row[4], (row[5])+1, 1)

        timing_one=time.time() #timing3
        ################################################## (changes here)
        var2=var
        if var2==f"qv'": var2='qv'

        if 'xf' in list(data[var2].coords): 
            x = parcel_read(slice,xh=row[0],var='x')/1000 
            xf = data['xf'].values
            which_x = [np.searchsorted(xf, x) - 1][0]
        elif 'xh' in list(data[var2].coords):
            x = parcel_read(slice,xh=row[0],var='x')/1000 
            xh = data['xh'].values
            which_x = [np.searchsorted(xh, x) - 1][0]
            
        if 'yf' in list(data[var2].coords): 
            y = parcel_read(slice,xh=row[0],var='y')/1000 
            yf = data['yf'].values
            which_y = [np.searchsorted(yf, y) - 1][0]
        elif 'yh' in list(data[var2].coords): 
            y = parcel_read(slice,xh=row[0],var='y')/1000 
            yh = data['yh'].values
            which_y = [np.searchsorted(yh, y) - 1][0]

        if 'zf' in list(data[var2].coords): 
            z = parcel_read(slice,xh=row[0],var='z')/1000 
            zf = data['zf'].values
            which_z = [np.searchsorted(zf, z) - 1][0]
        elif 'zh' in list(data[var2].coords): 
            z = parcel_read(slice,xh=row[0],var='z')/1000 
            zh = data['zh'].values
            which_z = [np.searchsorted(zh, z) - 1][0]
        ##################################################
        timing_two=time.time() #*timing3
        timing[3].append(timing_two-timing_one) #*timing3
    
        for t_ind in range((n+2)+1):

            timing_one=time.time() #timing4
            ################################################## (changes here)
            if 'xf' in list(data[var2].coords): 
                xbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_x[t_ind]]]
                xbox=[arr[(arr >= 0) & (arr < len(data['xf']))] for arr in xbox][0]
            elif 'xh' in list(data[var2].coords): 
                xbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_x[t_ind]]]
                xbox=[arr[(arr >= 0) & (arr < len(data['xh']))] for arr in xbox][0]

            if 'yh' in list(data[var2].coords): 
                ybox = np.mod([np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_y[t_ind]]],len(data['yh']))[0]
            elif 'yf' in list(data[var2].coords): 
                ybox = np.mod([np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_y[t_ind]]],len(data['yf']))[0]

            if 'yh' in list(data[var2].coords): 
                # zbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in which_z[t_ind]]
                # zbox=[arr[(arr >= 0) & (arr < len(data['zh']))] for arr in zbox]
                zbox=which_z[t_ind]
            elif 'yf' in list(data[var2].coords): 
                # zbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in which_z[t_ind]]
                # zbox=[arr[(arr >= 0) & (arr < len(data['zf']))] for arr in zbox]
                zbox=which_z[t_ind]
            ##################################################
            timing_two=time.time() #*timing4
            timing[4].append(timing_two-timing_one) #*timing4
            
            #getting data
            timing_one=time.time() #timing5
            #########################################################
            if var=='qvw':
                var1='qv'
                qvbox = read_var(slice,xbox,ybox,zbox,var1)
                qvbox=qvbox.isel(time=t_ind)
                qv=qvbox
                ####################    
                var2='w'
                wbox = read_var(slice,xbox,ybox,zbox,var2)
                # wbox=wbox.interp(zf=qvbox.zh,method='linear') #interpolation of w onto qv ********* need to fix
                wbox=wbox.isel(time=t_ind)
                w=wbox
                ####################    
                pdata=qv*w    
   
            elif var==f"qv'":
                box = read_var(slice,xbox,ybox,zbox,'qv')
                box=box.isel(time=t_ind)
                try:
                    pdata=box-box.mean(dim=('xh','yh'))
                except: 
                    pdata=box-box.mean(dim=('xh','yh')) 
            
            else:
                var1=var
                box = read_var(slice,xbox,ybox,zbox,var1)  
                box=box.isel(time=t_ind)
                pdata=box
            #########################################################
    
            #adds pdata at current time step to correct number timestep in pdata_mean
            if not np.any(pdata_mean[t_ind,:,:]): 
                pdata_mean[t_ind,:,:]=pdata.values
            elif len(xbox)!=len(ybox):
                #if the xbox doesn't match, remove this parcel from out_nz
                del_ind = np.where(out_nz[:, 0] == row[0])[0]
                out_nz = np.delete(out_nz, del_ind, axis=0)
            else:
                pdata_mean[t_ind,:,:]=(pdata_mean[t_ind,:,:] + pdata.values)
                
            timing_two=time.time() #*timing5
            timing[5].append(timing_two-timing_one) #*timing5
    
    pdata_mean/=len(out_nz[start_job:end_job+1]) 
    return pdata_mean
##################################################################################################################
arr={}
for i, (var, unit, long_name) in enumerate(zip(var_names, latex_units, long_names)): #better info
    pdata_mean=cllfc_mean(var=var,unit=unit, long_name=long_name)
    arr=data_save(var,arr,pdata_mean,savename=f'trajectory_{job_id}') #trajectory_dict=np.load('array.npy', allow_pickle=True)[()] 
    print(f'done with {var}')
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds") 

In [ ]:
#done seperately after all job_arrays completed

#combine all seperate job array data 
########################################################################
folder_path = 'arrays/'
files = [file for file in os.listdir(folder_path) 
         if file.startswith('trajectory_array') and file.endswith('.npy')]

combined_dict=np.load(folder_path+files[0], allow_pickle=True)[()] 
keys=list(trajectory_dict.keys())
for file in files[1:]:
    trajectory_dict=np.load(folder_path+file, allow_pickle=True)[()] 
    for key in keys:
        combined_dict[key]+=trajectory_dict[key]
for key in keys:
    combined_dict[key]/=len(files)
########################################################################


def plot_trajectory(pdata_mean, var, long_name, unit, zthresh, ns, n):
    contours=[] #colorbar fix 
    # pdata_mean=cllfc_mean(var,unit,long_name) #for non-job_array
    
    #plotting labels and parcel 
    #########################################################
    if var==f"qv'":
        cmap='bwr'
        norm = colors.TwoSlopeNorm(vcenter=0)
        vmin=pdata_mean.min();vmax=pdata_mean.max()
    elif var=='qv':
        cmap = 'Blues'
        norm = None
    else:
        cmap='viridis'
        norm = None
        vmin=pdata_mean.min();vmax=pdata_mean.max()
        
    num_plots=pdata_mean.shape[0]
    num_rows = np.ceil(np.sqrt(num_plots))
    ncols = int(np.ceil(num_plots / num_rows))
    fig_big = plt.figure(figsize=(18, 10))
    fig_big.suptitle(f'Mean 2x2 Box around Parcel from CL to LFC. z_top>{zthresh} km', fontsize=16)
    fig_big.text(0.5, 0.92, long_name, fontsize=12, ha='center', transform=fig_big.transFigure) #better info
    ax_big=gridspec.GridSpec(1, 2, figure=fig_big, width_ratios=[6,4])
    ax_left = gridspec.GridSpecFromSubplotSpec(int(num_rows), ncols, subplot_spec=ax_big[0])
    axs = ax_left.subplots()
    
    before=-1;after=1
    trange=np.arange(pdata_mean.shape[0])
    for i, t in enumerate(trange):
        prow = i // ncols
        pcol = i % ncols
        pdata=pdata_mean[t,:,:]

        # data_min = np.min(pdata_mean) #doesn't work for similar values
        # data_max = np.max(pdata_mean) 
        # levels=np.linspace(data_min, data_max, 10) 
        #(levels=levels)
    
        contour = axs[prow, pcol].contourf(pdata,cmap=cmap,norm=norm)#plot box data
        fig_big.colorbar(contour, ax=axs[prow, pcol])# add if want each subplot to have own colorbar
        contours.append(contour) #colorbar fix
        all_levels = np.concatenate([contour.levels for contour in contours]) #colorbar fix
        
        #adding grid lines
        xgrids=np.arange(0,(pdata.shape[1]-1),0.5)
        ygrids=np.arange(0,(pdata.shape[0]-1),0.5)
        
        xwhich=xgrids[np.where((xgrids % 1) == 0.5)]
        ywhich=ygrids[np.where((ygrids % 1) == 0.5)]
        for (xind, yind) in zip(xwhich, ywhich):
            axs[prow, pcol].axvline(xind, color='white',linestyle='--',label=None) #add grid
            axs[prow, pcol].axhline(yind, color='white',linestyle='--',label=None) #add grid
        #adding time titles
        axs[prow, pcol].set_title(f't={i+before}')
        lfc_ind=n
        if i==0:
            axs[prow, pcol].scatter(ns,ns,color='white',label='< CL')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
            axs[prow, pcol].set_ylabel('y (km)')
        if i==-before:
            axs[prow, pcol].scatter(ns,ns,color='orange',label='= CL')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
        if i in np.arange(-before+1,(lfc_ind)+1):
            axs[prow, pcol].scatter(ns,ns,color='orange',label='> CL')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
        if i==lfc_ind+1:
            axs[prow, pcol].scatter(ns,ns,color='red',label='≈ LFC+0-1km')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
        if i in np.arange(lfc_ind+2,len(trange)):
            axs[prow, pcol].scatter(ns,ns,color='white',label='> LFC')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
    
        #setting xticks and yticks
        xticks = axs[prow, pcol].get_xticks()
        xticks = [xtick for xtick in xticks if xtick.is_integer()] 
        axs[prow, pcol].set_xticks(xticks)
        axs[prow, pcol].set_xticklabels([int(xtick - ns) for xtick in xticks])
        yticks = axs[prow, pcol].get_yticks()
        yticks = [ytick for ytick in yticks if ytick.is_integer()] 
        axs[prow, pcol].set_yticks(yticks)
        axs[prow, pcol].set_yticklabels([int(ytick - ns) for ytick in yticks])

        if prow==(len(trange)-1)//ncols:
            axs[prow, pcol].set_xlabel('x (km)')
        else:
             axs[prow, pcol].set_xticks([])

        if pcol==0:
            axs[prow, pcol].set_ylabel('y (km)')
        else:
             axs[prow, pcol].set_yticks([])
            
    #########################################################        
    # # Add a single colorbar  
    # num_ticks=10 #colorbar fix
    # levels = np.linspace(all_levels.min(), all_levels.max(), num_ticks) #colorbar fix
    # cmap = contours[0].cmap #colorbar fix
    # mappable = cm.ScalarMappable(norm=colors.BoundaryNorm(levels, cmap.N), cmap=cmap)  #colorbar fix
    # colorbar = fig_big.colorbar(mappable, ax=axs.ravel().tolist()) #colorbar fix
    # colorbar.set_ticks(levels) #colorbar fix
    # colorbar.set_label(f'{var} {unit}', rotation=90) #better info
    
    ##################################################################################################################
    
    #appending data
    #########################################################
    lst,tl,tr,bl,br=[],[],[],[],[]
    for t_ind in np.arange(pdata_mean.shape[0]):
        #1km either side
        tlrange=[np.arange(0,(ns)+1),np.arange(0,(ns)+1)]
        trrange=[np.arange(0,(ns)+1),np.arange(ns,(2*ns)+1)]
        blrange=[np.arange(ns,(2*ns)+1),np.arange(0,(ns)+1)]
        brrange=[np.arange(ns,(2*ns)+1),np.arange(ns,(2*ns)+1)]
        
        temp=pdata_mean.copy()[t_ind,tlrange[0][0]:tlrange[0][-1]+1,tlrange[1][0]:tlrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        tl.append(np.nanmean(temp))
        
        temp=pdata_mean.copy()[t_ind,trrange[0][0]:trrange[0][-1]+1,trrange[1][0]:trrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        tr.append(np.nanmean(temp))
    
        lst.append(pdata_mean[t_ind,ns,ns]) #center
        
        temp=pdata_mean.copy()[t_ind,blrange[0][0]:blrange[0][-1]+1,blrange[1][0]:blrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        bl.append(np.nanmean(temp))
        
        temp=pdata_mean.copy()[t_ind,brrange[0][0]:brrange[0][-1]+1,brrange[1][0]:brrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        br.append(np.nanmean(temp))
        
        
    #########################################################
    
    
    ################################################################################
    ax_right = gridspec.GridSpecFromSubplotSpec(4, 3, subplot_spec=ax_big[1])
    
    axs_extra = fig_big.add_subplot(ax_right[0:2, 0:2])
    ###################################################
    axs_extra.plot(tl, color='green', label='TL parcel',zorder=2)
    axs_extra.plot(tr, color='blue', label='TR parcel',zorder=2)
    axs_extra.plot(lst, color='black', label='at parcel',zorder=2)
    axs_extra.plot(bl, color='purple', label='BL parcel',zorder=2)
    axs_extra.plot(br, color='brown', label='BR parcel',zorder=2)
    ###################################################
    axs_extra.axvline(0 - before, color='orange',zorder=1)#, label='at CL')
    axs_extra.axvline(n+after, color='red',zorder=1)#, label='at LFC')
    axs_extra.set_xlabel('t (5 mins)');axs_extra.set_ylabel(f'{var} {unit}') #better info
    axs_extra.legend(loc='upper center')
    
    #fixing ticks and labels
    xticks = axs_extra.get_xticks()
    xtick_labels = [str(int(x)-1) for x in xticks]
    from matplotlib.ticker import FixedLocator
    axs_extra.xaxis.set_major_locator(FixedLocator(xticks))
    axs_extra.set_xticklabels(xtick_labels)
    
    ################################################################################
    ax = fig_big.add_subplot(ax_right[0, 2:])
    from matplotlib.patches import Rectangle,Polygon
    
    ax.scatter(2,2,color='red',label='parcel',zorder=3)
    
    #patches
    plt.xlim([0,4])
    plt.ylim([0,4])
    black_rect = Rectangle((0,0), 4, 4, facecolor='black')
    ax.add_patch(black_rect) 
    green_rect = Rectangle((0.5,2.5), 1.5, 1, facecolor='green',zorder=2)
    ax.add_patch(green_rect)
    green_rect = Rectangle((0.5,2), 1, 1.5, facecolor='green',zorder=2)
    ax.add_patch(green_rect)
    blue_rect = Rectangle((2,2.5), 1.5, 1, facecolor='blue',zorder=2)
    ax.add_patch(blue_rect) 
    blue_rect = Rectangle((2.5,2), 1, 1.5, facecolor='blue',zorder=2)
    ax.add_patch(blue_rect) 
    purple_rect = Rectangle((0.5,0.5), 1, 1.5, facecolor='purple',zorder=2)
    ax.add_patch(purple_rect)
    purple_rect = Rectangle((0.5,0.5), 1.5, 1, facecolor='purple',zorder=2)
    ax.add_patch(purple_rect)
    brown_rect = Rectangle((2,0.5), 1.5, 1, facecolor='brown',zorder=2)
    ax.add_patch(brown_rect)
    brown_rect = Rectangle((2.5,0.5), 1, 1.5, facecolor='brown',zorder=2)
    ax.add_patch(brown_rect)
    
    #grid lines
    xwhich=xgrids[np.where((xgrids % 1) == 0.5)]
    ywhich=ygrids[np.where((ygrids % 1) == 0.5)]
    for (xind, yind) in zip(xwhich, ywhich):
        ax.axvline(xind, color='white',linestyle='--',label=None) #add grid
        ax.axhline(yind, color='white',linestyle='--',label=None) #add grid
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper center',fontsize='small',framealpha=0.5)

    ################################################################################
    #saving
    folder_path='/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/trajectories/'
    # plt.savefig(os.path.join(folder_path, f'lagrangian_{var}_mean.jpg'),dpi=300) 
    # plt.close()
    return pdata_mean

for i, (var, unit, long_name) in enumerate(zip(var_names, latex_units, long_names)): #better info
    pdata_mean=combined_dict[var+'_pdata_mean'];zthresh=5 #***
    kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1);ns=2*kms #finds how many x grids is 1 km
    n=int(30/times[1]) #number of times to split CL-LFC trajectory into

    pdata_mean=plot_trajectory(pdata_mean, var, long_name, unit, zthresh, ns, n)
    print(f'done with {var}')

In [ ]:
#Plotting data 2x2x2 box (zmean) around parcel from CL to LFC *Mean of convective minus nonconvective parcels* #ADDED JOB ARRAY
#
# (1) runs through all parcel 
# (2) time slice from CL-LFC is found
# (3) time slice is used to find the parcels x,y, and z grid locations (interpolated to 7+-1 times)
# (4) for each time, a 2x2 km box around parcel is added to empty array for that time
# (5) does same thing for non-convective parcels and subtracts from convective parcels
# (6) creates figures
#
start_time=time.time()
if run==True:
    out_nz2=out_nz.copy()
    zthresh=5
    [out_nz,extendrange]=threshold(zthresh)
    run=False

def data_save(var,arr,pdata_mean,savename):
    folder_path='/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/diff_trajectories/'
    arr[var+'_pdata_mean'] = pdata_mean
    np.save(folder_path+savename+'_array.npy', arr)
    return arr

#finding other parcels 
where = np.where(np.isin(out_nz2[:, 0], out_nz[:, 0], invert=True))
out_nz_less=out_nz2[where]

def cllfc_mean(var,unit,long_name):
    global out_nz, out_nz_less #for diff mean
    contours=[] #colorbar fix 
   
    def read_var(slice, x, y, z, var):
        data['time']=np.arange(len(data['time']))
        coord_map = {
            'xh': 'xf',
            'yh': 'yf',
            'zh': 'zf'
        }
        
        var_coords = data[var].coords
        coords=['xh', 'yh', 'zh'] 
        for ind, coord in enumerate(coords):
            if coord not in var_coords:
                coords[ind] = coord_map[coord]
        indices = {}
        indices[coords[0]] = x
        indices[coords[1]] = y
        indices[coords[2]] = z
        # output=data[var].isel(**{coord: indices[coord] for coord in coords}) #old 
        ############################
        output=data[var].isel(**{coord: indices[coord] for coord in coords[0:2]}) #splitting horizontal and vertical
        output=output.isel(**{coord: indices[coord] for coord in [coords[2]]}) #splitting horizontal and vertical
        
        #interpolation
        t1=slice[0];t2=slice[-1]
        even=(t2-t1)/n
        new_t=np.linspace(t1, t2, n+1)
        new_t=np.insert(new_t,0,new_t[0]-even)
        new_t=np.append(new_t,new_t[-1]+even)
        output=output.interp(time=new_t,method='linear')
        
        return output

    def parcel_read(slice,xh,var):
        parcel['time']=np.arange(len(parcel['time']))
        output=parcel[var].isel(xh=row[0])
        #interpolation
        t1=slice[0];t2=slice[-1]
        even=(t2-t1)/n
        new_t=np.linspace(t1, t2, n+1)
        new_t=np.insert(new_t,0,new_t[0]-even)
        new_t=np.append(new_t,new_t[-1]+even)
        output=output.interp(time=new_t,method='linear').values
        return output
    
    zlev=2
    kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #finds how many x grids is 1 km
    ns=2*kms
    n=int(30/times[1]) #number of times to split CL-LFC trajectory into
    
    pdata_mean=np.zeros(((n+2)+1,2*ns+1,2*ns+1))
    for prnt, row in enumerate(out_nz):
        if np.mod(prnt,25)==0: print(f'{prnt}/{len(out_nz)} parcels done') #TESTING
            
        slice= np.arange(row[4], (row[5])+1, 1)

        ################################################## (changes here)
        var2=var
        if var2==f"qv'": var2='qv'

        if 'xf' in list(data[var2].coords): 
            x = parcel_read(slice,xh=row[0],var='x')/1000 
            xf = data['xf'].values
            which_x = [np.searchsorted(xf, x) - 1][0]
        elif 'xh' in list(data[var2].coords):
            x = parcel_read(slice,xh=row[0],var='x')/1000 
            xh = data['xh'].values
            which_x = [np.searchsorted(xh, x) - 1][0]
            
        if 'yf' in list(data[var2].coords): 
            y = parcel_read(slice,xh=row[0],var='y')/1000 
            yf = data['yf'].values
            which_y = [np.searchsorted(yf, y) - 1][0]
        elif 'yh' in list(data[var2].coords): 
            y = parcel_read(slice,xh=row[0],var='y')/1000 
            yh = data['yh'].values
            which_y = [np.searchsorted(yh, y) - 1][0]

        if 'zf' in list(data[var2].coords): 
            z = parcel_read(slice,xh=row[0],var='z')/1000 
            zf = data['zf'].values
            which_z = [np.searchsorted(zf, z) - 1][0]
        elif 'zh' in list(data[var2].coords): 
            z = parcel_read(slice,xh=row[0],var='z')/1000 
            zh = data['zh'].values
            which_z = [np.searchsorted(zh, z) - 1][0]
        ##################################################
    
        for t_ind in range((n+2)+1):

            ################################################## (changes here)
            if 'xf' in list(data[var2].coords): 
                xbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_x[t_ind]]]
                xbox=[arr[(arr >= 0) & (arr < len(data['xf']))] for arr in xbox][0]
            elif 'xh' in list(data[var2].coords): 
                xbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_x[t_ind]]]
                xbox=[arr[(arr >= 0) & (arr < len(data['xh']))] for arr in xbox][0]

            if 'yh' in list(data[var2].coords): 
                ybox = np.mod([np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_y[t_ind]]],len(data['yh']))[0]
            elif 'yf' in list(data[var2].coords): 
                ybox = np.mod([np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_y[t_ind]]],len(data['yf']))[0]

            if 'yh' in list(data[var2].coords): 
                # zbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in which_z[t_ind]]
                # zbox=[arr[(arr >= 0) & (arr < len(data['zh']))] for arr in zbox]
                zbox=which_z[t_ind]
            elif 'yf' in list(data[var2].coords): 
                # zbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in which_z[t_ind]]
                # zbox=[arr[(arr >= 0) & (arr < len(data['zf']))] for arr in zbox]
                zbox=which_z[t_ind]
            ##################################################
            
            #getting data
            #########################################################
            if var=='qvw':
                var1='qv'
                qvbox = read_var(slice,xbox,ybox,zbox,var1)
                qvbox=qvbox.isel(time=t_ind)
                qv=qvbox
                ####################    
                var2='w'
                wbox = read_var(slice,xbox,ybox,zbox,var2)
                # wbox=wbox.interp(zf=qvbox.zh,method='linear') #interpolation of w onto qv ********* need to fix
                wbox=wbox.isel(time=t_ind)
                w=wbox
                ####################    
                pdata=qv*w    
   
            elif var==f"qv'":
                box = read_var(slice,xbox,ybox,zbox,'qv')
                box=box.isel(time=t_ind)
                try:
                    pdata=box-box.mean(dim=('xh','yh'))
                except: 
                    pdata=box-box.mean(dim=('xh','yh')) 
            
            else:
                var1=var
                box = read_var(slice,xbox,ybox,zbox,var1)  
                box=box.isel(time=t_ind)
                pdata=box
            #########################################################
    
    
            #adds pdata at current time step to correct number timestep in pdata_mean
            if not np.any(pdata_mean[t_ind,:,:]): 
                pdata_mean[t_ind,:,:]=pdata.values
            elif len(xbox)!=len(ybox):
                #if the xbox doesn't match, remove this parcel from out_nz
                del_ind = np.where(out_nz[:, 0] == row[0])[0]
                out_nz = np.delete(out_nz, del_ind, axis=0)
            else:
                pdata_mean[t_ind,:,:]=(pdata_mean[t_ind,:,:] + pdata.values)
    
    pdata_mean/=len(out_nz) 
    ##################################################################################################################
    #for diff mean
    pdata_mean1=pdata_mean.copy() #for diff mean
    pdata_mean=np.zeros(((n+2)+1,2*ns+1,2*ns+1))
    for prnt, row in enumerate(out_nz_less[start_job:end_job+1]):
        if np.mod(prnt,25)==0: print(f'{prnt}/{len(out_nz_less)} less parcels done') #TESTING
            
        slice= np.arange(row[4], (row[5])+1, 1)

        ################################################## (changes here)
        var2=var
        if var2==f"qv'": var2='qv'

        if 'xf' in list(data[var2].coords): 
            x = parcel_read(slice,xh=row[0],var='x')/1000 
            xf = data['xf'].values
            which_x = [np.searchsorted(xf, x) - 1][0]
        elif 'xh' in list(data[var2].coords):
            x = parcel_read(slice,xh=row[0],var='x')/1000 
            xh = data['xh'].values
            which_x = [np.searchsorted(xh, x) - 1][0]
            
        if 'yf' in list(data[var2].coords): 
            y = parcel_read(slice,xh=row[0],var='y')/1000 
            yf = data['yf'].values
            which_y = [np.searchsorted(yf, y) - 1][0]
        elif 'yh' in list(data[var2].coords): 
            y = parcel_read(slice,xh=row[0],var='y')/1000 
            yh = data['yh'].values
            which_y = [np.searchsorted(yh, y) - 1][0]

        if 'zf' in list(data[var2].coords): 
            z = parcel_read(slice,xh=row[0],var='z')/1000 
            zf = data['zf'].values
            which_z = [np.searchsorted(zf, z) - 1][0]
        elif 'zh' in list(data[var2].coords): 
            z = parcel_read(slice,xh=row[0],var='z')/1000 
            zh = data['zh'].values
            which_z = [np.searchsorted(zh, z) - 1][0]
        ##################################################
    
        for t_ind in range((n+2)+1):

            ################################################## (changes here)
            if 'xf' in list(data[var2].coords): 
                xbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_x[t_ind]]]
                xbox=[arr[(arr >= 0) & (arr < len(data['xf']))] for arr in xbox][0]
            elif 'xh' in list(data[var2].coords): 
                xbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_x[t_ind]]]
                xbox=[arr[(arr >= 0) & (arr < len(data['xh']))] for arr in xbox][0]

            if 'yh' in list(data[var2].coords): 
                ybox = np.mod([np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_y[t_ind]]],len(data['yh']))[0]
            elif 'yf' in list(data[var2].coords): 
                ybox = np.mod([np.arange(ind-ns, (ind+ns)+1, 1) for ind in [which_y[t_ind]]],len(data['yf']))[0]

            if 'yh' in list(data[var2].coords): 
                # zbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in which_z[t_ind]]
                # zbox=[arr[(arr >= 0) & (arr < len(data['zh']))] for arr in zbox]
                zbox=which_z[t_ind]
            elif 'yf' in list(data[var2].coords): 
                # zbox = [np.arange(ind-ns, (ind+ns)+1, 1) for ind in which_z[t_ind]]
                # zbox=[arr[(arr >= 0) & (arr < len(data['zf']))] for arr in zbox]
                zbox=which_z[t_ind]
            ##################################################
            
            #getting data
            #########################################################
            if var=='qvw':
                var1='qv'
                qvbox = read_var(slice,xbox,ybox,zbox,var1)
                qvbox=qvbox.isel(time=t_ind)
                qv=qvbox
                ####################    
                var2='w'
                wbox = read_var(slice,xbox,ybox,zbox,var2)
                # wbox=wbox.interp(zf=qvbox.zh,method='linear') #interpolation of w onto qv ********* need to fix
                wbox=wbox.isel(time=t_ind)
                w=wbox
                ####################    
                pdata=qv*w    
   
            elif var==f"qv'":
                box = read_var(slice,xbox,ybox,zbox,'qv')
                box=box.isel(time=t_ind)
                try:
                    pdata=box-box.mean(dim=('xh','yh'))
                except: 
                    pdata=box-box.mean(dim=('xh','yh')) 
            
            else:
                var1=var
                box = read_var(slice,xbox,ybox,zbox,var1)  
                box=box.isel(time=t_ind)
                pdata=box
            #########################################################
    
    
            #adds pdata at current time step to correct number timestep in pdata_mean
            if not np.any(pdata_mean[t_ind,:,:]): 
                pdata_mean[t_ind,:,:]=pdata.values
            elif len(xbox)!=len(ybox): #for diff mean
                #if the xbox doesn't match, remove this parcel from out_nz
                del_ind = np.where(out_nz_less[:, 0] == row[0])[0]
                out_nz_less = np.delete(out_nz_less, del_ind, axis=0)
            else:
                pdata_mean[t_ind,:,:]=(pdata_mean[t_ind,:,:] + pdata.values)
    
    pdata_mean/=len(out_nz_less[start_job:end_job+1]) #for diff mean
    pdata_mean=pdata_mean1-pdata_mean #for diff mean
    return pdata_mean
    ##################################################################################################################

arr={}
for var, unit, long_name in zip(var_names, latex_units, long_names): #better info
    if i in np.arange(start_job,(end_job)+1): #for job array
        pdata_mean=cllfc_mean(var=var,unit=unit, long_name=long_name)
        arr=data_save(var,arr,pdata_mean,savename=f'diff_trajectory_{job_id}') #trajectory_dict=np.load('array.npy', allow_pickle=True)[()] 
        print(f'done with {var}')
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds") 

In [ ]:
# done seperately after all job_arrays completed

#combine all seperate job array data 
########################################################################
folder_path = 'arrays/'
files = [file for file in os.listdir(folder_path) 
         if file.startswith('diff_trajectory_array') and file.endswith('.npy')]

combined_dict=np.load(folder_path+files[0], allow_pickle=True)[()] 
keys=list(trajectory_dict.keys())
for file in files[1:]:
    trajectory_dict=np.load(folder_path+file, allow_pickle=True)[()] 
    for key in keys:
        combined_dict[key]+=trajectory_dict[key]
for key in keys:
    combined_dict[key]/=len(files)
########################################################################

def plot_trajectory(pdata_mean, var, long_name, unit, zthresh, ns, n):   
    contours=[] #colorbar fix 
    #plotting labels and parcel 
    #########################################################
    if var==f"qv'":
        cmap='bwr'
        norm = colors.TwoSlopeNorm(vcenter=0)
        vmin=pdata_mean.min();vmax=pdata_mean.max()
    elif var=='qv':
        cmap = 'Blues'
        norm = None
    else:
        cmap='viridis'
        norm = None
        vmin=pdata_mean.min();vmax=pdata_mean.max()
        
    num_plots=pdata_mean.shape[0]
    num_rows = np.ceil(np.sqrt(num_plots))
    ncols = int(np.ceil(num_plots / num_rows))
    fig_big = plt.figure(figsize=(18, 10))
    fig_big.suptitle(f'Diff Mean 2x2 Box around Parcel from CL to LFC. z_top>{zthresh} km', fontsize=16) #for diff mean
    fig_big.text(0.5, 0.92, long_name, fontsize=12, ha='center', transform=fig_big.transFigure) #better info
    ax_big=gridspec.GridSpec(1, 2, figure=fig_big, width_ratios=[6,4])
    ax_left = gridspec.GridSpecFromSubplotSpec(int(num_rows), ncols, subplot_spec=ax_big[0])
    axs = ax_left.subplots()
    
    trange=np.arange(pdata_mean.shape[0])
    before=-1;after=1
    for i, t in enumerate(trange):
        prow = i // ncols
        pcol = i % ncols
        pdata=pdata_mean[t,:,:]

        # data_min = np.min(pdata_mean) #doesn't work for similar values
        # data_max = np.max(pdata_mean) 
        # levels=np.linspace(data_min, data_max, 10) 
        #(levels=levels)
    
        contour = axs[prow, pcol].contourf(pdata,cmap=cmap,norm=norm)#plot box data
        fig_big.colorbar(contour, ax=axs[prow, pcol])# add if want each subplot to have own colorbar
        contours.append(contour) #colorbar fix
        all_levels = np.concatenate([contour.levels for contour in contours]) #colorbar fix
        
        #adding grid lines
        xgrids=np.arange(0,(pdata.shape[1]-1),0.5)
        ygrids=np.arange(0,(pdata.shape[0]-1),0.5)
        
        xwhich=xgrids[np.where((xgrids % 1) == 0.5)]
        ywhich=ygrids[np.where((ygrids % 1) == 0.5)]
        for (xind, yind) in zip(xwhich, ywhich):
            axs[prow, pcol].axvline(xind, color='white',linestyle='--',label=None) #add grid
            axs[prow, pcol].axhline(yind, color='white',linestyle='--',label=None) #add grid
        #adding time titles
        axs[prow, pcol].set_title(f't={i+before}')
        lfc_ind=n
        if i==0:
            axs[prow, pcol].scatter(ns,ns,color='white',label='< CL')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
            axs[prow, pcol].set_ylabel('y (km)')
        if i==-before:
            axs[prow, pcol].scatter(ns,ns,color='orange',label='= CL')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
        if i in np.arange(-before+1,(lfc_ind)+1):
            axs[prow, pcol].scatter(ns,ns,color='orange',label='> CL')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
        if i==lfc_ind+1:
            axs[prow, pcol].scatter(ns,ns,color='red',label='≈ LFC+0-1km')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
        if i in np.arange(lfc_ind+2,len(trange)):
            axs[prow, pcol].scatter(ns,ns,color='white',label='> LFC')
            axs[prow, pcol].legend(loc='upper right',framealpha=0.5)
    
        #setting xticks and yticks
        xticks = axs[prow, pcol].get_xticks()
        xticks = [xtick for xtick in xticks if xtick.is_integer()] 
        axs[prow, pcol].set_xticks(xticks)
        axs[prow, pcol].set_xticklabels([int(xtick - ns) for xtick in xticks])
        yticks = axs[prow, pcol].get_yticks()
        yticks = [ytick for ytick in yticks if ytick.is_integer()] 
        axs[prow, pcol].set_yticks(yticks)
        axs[prow, pcol].set_yticklabels([int(ytick - ns) for ytick in yticks])

        if prow==(len(trange)-1)//ncols:
            axs[prow, pcol].set_xlabel('x (km)')
        else:
             axs[prow, pcol].set_xticks([])

        if pcol==0:
            axs[prow, pcol].set_ylabel('y (km)')
        else:
             axs[prow, pcol].set_yticks([])
            
    #########################################################        
    # # Add a single colorbar  
    # num_ticks=10 #colorbar fix
    # levels = np.linspace(all_levels.min(), all_levels.max(), num_ticks) #colorbar fix
    # cmap = contours[0].cmap #colorbar fix
    # mappable = cm.ScalarMappable(norm=colors.BoundaryNorm(levels, cmap.N), cmap=cmap)  #colorbar fix
    # colorbar = fig_big.colorbar(mappable, ax=axs.ravel().tolist()) #colorbar fix
    # colorbar.set_ticks(levels) #colorbar fix
    # colorbar.set_label(f'{var} {unit}', rotation=90) #better info
    
    ##################################################################################################################
    
    #appending data
    #########################################################
    lst,tl,tr,bl,br=[],[],[],[],[]
    for t_ind in np.arange(pdata_mean.shape[0]):
        #1km either side
        tlrange=[np.arange(0,(ns)+1),np.arange(0,(ns)+1)]
        trrange=[np.arange(0,(ns)+1),np.arange(ns,(2*ns)+1)]
        blrange=[np.arange(ns,(2*ns)+1),np.arange(0,(ns)+1)]
        brrange=[np.arange(ns,(2*ns)+1),np.arange(ns,(2*ns)+1)]
        
        temp=pdata_mean.copy()[t_ind,tlrange[0][0]:tlrange[0][-1]+1,tlrange[1][0]:tlrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        tl.append(np.nanmean(temp))
        
        temp=pdata_mean.copy()[t_ind,trrange[0][0]:trrange[0][-1]+1,trrange[1][0]:trrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        tr.append(np.nanmean(temp))
    
        lst.append(pdata_mean[t_ind,ns,ns]) #center
        
        temp=pdata_mean.copy()[t_ind,blrange[0][0]:blrange[0][-1]+1,blrange[1][0]:blrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        bl.append(np.nanmean(temp))
        
        temp=pdata_mean.copy()[t_ind,brrange[0][0]:brrange[0][-1]+1,brrange[1][0]:brrange[1][-1]+1]
        temp[ns-1,ns-1]=np.nan
        br.append(np.nanmean(temp))
          
    #########################################################
    
    
    ################################################################################
    ax_right = gridspec.GridSpecFromSubplotSpec(4, 3, subplot_spec=ax_big[1])
    
    axs_extra = fig_big.add_subplot(ax_right[0:2, 0:2])
    ###################################################
    axs_extra.plot(tl, color='green', label='TL parcel',zorder=2)
    axs_extra.plot(tr, color='blue', label='TR parcel',zorder=2)
    axs_extra.plot(lst, color='black', label='at parcel',zorder=2)
    axs_extra.plot(bl, color='purple', label='BL parcel',zorder=2)
    axs_extra.plot(br, color='brown', label='BR parcel',zorder=2)
    ###################################################
    axs_extra.axvline(0 - before, color='orange',zorder=1)#, label='at CL')
    axs_extra.axvline(n+after, color='red',zorder=1)#, label='at LFC')
    axs_extra.set_xlabel('t (5 mins)');axs_extra.set_ylabel(f'{var} {unit}') #better info
    axs_extra.legend(loc='upper center')
    
    #fixing ticks and labels
    xticks = axs_extra.get_xticks()
    xtick_labels = [str(int(x)-1) for x in xticks]
    from matplotlib.ticker import FixedLocator
    axs_extra.xaxis.set_major_locator(FixedLocator(xticks))
    axs_extra.set_xticklabels(xtick_labels)
    
    ################################################################################
    ax = fig_big.add_subplot(ax_right[0, 2:])
    from matplotlib.patches import Rectangle,Polygon
    
    ax.scatter(2,2,color='red',label='parcel',zorder=3)
    
    #patches
    plt.xlim([0,4])
    plt.ylim([0,4])
    black_rect = Rectangle((0,0), 4, 4, facecolor='black')
    ax.add_patch(black_rect) 
    green_rect = Rectangle((0.5,2.5), 1.5, 1, facecolor='green',zorder=2)
    ax.add_patch(green_rect)
    green_rect = Rectangle((0.5,2), 1, 1.5, facecolor='green',zorder=2)
    ax.add_patch(green_rect)
    blue_rect = Rectangle((2,2.5), 1.5, 1, facecolor='blue',zorder=2)
    ax.add_patch(blue_rect) 
    blue_rect = Rectangle((2.5,2), 1, 1.5, facecolor='blue',zorder=2)
    ax.add_patch(blue_rect) 
    purple_rect = Rectangle((0.5,0.5), 1, 1.5, facecolor='purple',zorder=2)
    ax.add_patch(purple_rect)
    purple_rect = Rectangle((0.5,0.5), 1.5, 1, facecolor='purple',zorder=2)
    ax.add_patch(purple_rect)
    brown_rect = Rectangle((2,0.5), 1.5, 1, facecolor='brown',zorder=2)
    ax.add_patch(brown_rect)
    brown_rect = Rectangle((2.5,0.5), 1, 1.5, facecolor='brown',zorder=2)
    ax.add_patch(brown_rect)
    
    #grid lines
    xwhich=xgrids[np.where((xgrids % 1) == 0.5)]
    ywhich=ygrids[np.where((ygrids % 1) == 0.5)]
    for (xind, yind) in zip(xwhich, ywhich):
        ax.axvline(xind, color='white',linestyle='--',label=None) #add grid
        ax.axhline(yind, color='white',linestyle='--',label=None) #add grid
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper center',fontsize='small',framealpha=0.5)

    ################################################################################
    #saving
    folder_path='/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/diff_trajectories/'
    plt.savefig(os.path.join(folder_path, f'lagrangian_{var}_diff_mean.jpg'),dpi=300) 
    plt.close()
    return pdata_mean

for var, unit, long_name in zip(var_names, latex_units, long_names): #better info
    pdata_mean=combined_dict[var+'_pdata_mean'];zthresh=5 #***
    kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1);ns=2*kms #finds how many x grids is 1 km
    n=int(30/times[1]) #number of times to split CL-LFC trajectory into
    
    pdata_mean=plot_trajectory(pdata_mean, var, long_name, unit, zthresh, ns, n)
    print(f'done with {var}')
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds") 

In [ ]:
#################################

In [ ]:
## Converts all figures to PDF
######################################################################################################################################################
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from PIL import Image
import os

def jpg_to_pdf(input_folder, output_pdf):
    # Get a list of all JPG files in the input folder
    jpg_files = [file for file in os.listdir(input_folder) if file.endswith('.jpg')]
    
    # Create a PDF canvas
    c = canvas.Canvas(output_pdf, pagesize=letter)

    # Loop through each JPG file and add it to the PDF
    for jpg_file in jpg_files:
        # Open the JPG image using PIL
        img = Image.open(os.path.join(input_folder, jpg_file))

        # Calculate the aspect ratio to maintain image proportions
        width, height = img.size
        aspect_ratio = width / height

        # Add the image to the PDF
        c.setPageSize((width, height))
        c.drawInlineImage(os.path.join(input_folder, jpg_file), 0, 0, width=width, height=height)

        # Add a new page for the next image
        c.showPage()

    # Save the PDF
    c.save()

# #for trajectory singles
# zthresh=5
# input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/singles' 
# output_pdf = f'/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/singles/trajectory_singles>{zthresh}km.pdf'
# jpg_to_pdf(input_folder, output_pdf)

#for trajectory means
zthresh=5
input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/trajectories' 
output_pdf = f'/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/trajectories/trajectory_means>{zthresh}km.pdf'
jpg_to_pdf(input_folder, output_pdf)

#for trajectory diff means
zthresh=5
input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/diff_trajectoriess' 
output_pdf = f'/mnt/lustre/koa/koastore/torri_group/air_directory/job_array/plots/diff_trajectoriess/trajectory_diff_means>{zthresh}km.pdf'
jpg_to_pdf(input_folder, output_pdf)

In [ ]:
#scatter plots TESTING
def parcel_analysis(ind,var,extendrange,out_nz): #ind is number SBZ parcel, var is the variable

    time_range=np.arange(out_nz[ind,4],out_nz[ind,5]+1+int(np.max(extendrange)))
    time_range=time_range[time_range<len(data['time'])]
    analysis=parcel[var].isel(time=time_range,xh=out_nz[ind,0]).values
    if var=='w': analysis/=1000
    return analysis

variables = {
    'w': Variable('w','w',r'$m s^{-1}$',[]),
    'b': Variable('b','buoyancy',r'$m s^{-1} s^{-1}$',[]),
    'qv': Variable('qv','qv',r'$kg kg^{-1}$',[]),
    'zvort': Variable('zvort','z vorticity',r'$s^{-1}$',[])
}


[out_nz6,extendrange6]=threshold(6)
[out_nz8,extendrange8]=threshold(8)
[out_nz10,extendrange10]=threshold(10)
fig, axs = plt.subplots(2, 2, figsize=(12, 8))  # 2x2 subplots
for i, (key, var) in enumerate(variables.items()):
    row = i // 2; col = i % 2

    for ind in range(0, len(out_nz6)):
        var.vals.extend(parcel_analysis(ind,var.name,extendrange6,out_nz6))
    all_data=var.vals
    axs[row, col].scatter(np.arange(len(all_data)),all_data,s=1,color='black',alpha=0.5)
    axs[row,col].axhline(np.mean(all_data),color='black',zorder=1,label=f'> 6 km {np.mean(all_data):.6f}')
    var.vals=[]
    
    for ind in range(0, len(out_nz8)):
        var.vals.extend(parcel_analysis(ind,var.name,extendrange8,out_nz8))
    all_data=var.vals
    axs[row, col].scatter(np.arange(len(all_data)),all_data,s=1,color='blue',alpha=0.4)
    axs[row,col].axhline(np.mean(all_data),color='blue',zorder=1,label=f'> 8 km {np.mean(all_data):.6f}')
    var.vals=[]

    for ind in range(0, len(out_nz10)):
        var.vals.extend(parcel_analysis(ind,var.name,extendrange10,out_nz10))
    all_data=var.vals
    axs[row, col].scatter(np.arange(len(all_data)),all_data,s=1,color='red',alpha=0.3)
    axs[row,col].axhline(np.mean(all_data),color='red',zorder=1,label=f'> 10 km {np.mean(all_data):.6f}')
    var.vals=[]

    ##############################################################################
    axs[row, col].set_title(var.title)
    axs[row, col].set_xlabel('time (5 min intervals)')
    axs[row, col].set_ylabel(var.ylabs) 
    axs[row,col].legend()
fig.suptitle(f'lined up properties of parcels from CL to LFC. z_LFC>{zthresh} km')
plt.tight_layout()

In [ ]:
#Plotting Selected Parcel Individual Plots

zthresh=8
[out_nz,extendrange]=threshold(zthresh)

folder_path = f'/mnt/lustre/koa/koastore/torri_group/air_directory/trajectories_062217_>{zthresh}km' #***
import os; os.makedirs(folder_path, exist_ok=True)
start_time=time.time()
for k in range(len(out_nz)):
    print(f'parcel number {out_nz[k:k+1,0]}')
    for xh in out_nz[k:k+1,0]:
        #plotting full z trajectory
        z=parcel['z'].isel(xh=xh).values/1000; 
        channel_aspect_ratio = 3
        plt.figure(figsize=(10, 10/channel_aspect_ratio)) 
        plt.plot(range(len(z)),z,label='z')

        #coloring line by x location
        lst=[]
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            lst.append(which_x)
        colors = plt.cm.viridis  # Choose a colormap, here using 'viridis'
        norm = plt.Normalize(vmin=min(lst), vmax=max(lst))
        plt.scatter(range(len(z)), z, c=lst, cmap=colors, norm=norm,s=4)
        plt.colorbar(label='x grid')

        #data for CL parcel location
        ind=np.where(out_nz[:,0]==xh)[0][0]
        z_CL=parcel['z'].isel(time=out_nz[ind][4],xh=xh).values/1000;

        #data for LFC parcel location
        xloc=parcel['x'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000; yloc=parcel['y'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
        xf=data['xf'].values; which_x=np.searchsorted(xf,xloc)-1; yf=data['yf'].values; which_y=np.searchsorted(yf,yloc)-1; 
        z=parcel['z'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
        z_lfc=data['lfc'].isel(time=out_nz[ind,5],xh=which_x,yh=which_y).values/1000

        #plotting points for CL and LFC parcel location
        plt.scatter(out_nz[ind][4],z_CL,color='red',zorder=10,label='Convergence Max') #plotting max CL point
        plt.scatter(out_nz[ind][5],z,color='orange',zorder=10,label='LFC') #plotting LFC points
        plt.legend()

        #add some text with the velocity at the CL
        x_center = sum(plt.xlim()) / 2
        y_center = sum(plt.ylim()) / 2
        text='CL w: '+str(parcel['w'].isel(time=out_nz[ind][4],xh=xh).values)
        plt.text(x_center, y_center,text, ha='center', va='center')

        #add some text with the velocity at the LFC
        x_center = sum(plt.xlim()) / 3
        y_center = sum(plt.ylim()) / 3
        text='CL w: '+str(parcel['w'].isel(time=out_nz[ind][5],xh=xh).values)
        plt.text(x_center, y_center,text, ha='center', va='center')

        #plotting the line for LFC at all timesteps
        lst1,lst2=[],[]
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            y=parcel['y'].isel(time=t,xh=xh).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
            lfc=data['lfc'].isel(time=t,xh=which_x,yh=which_y).values/1000
            lcl=data['lcl'].isel(time=t,xh=which_x,yh=which_y).values/1000
            lst1.append(lfc);lst2.append(lcl)
        plt.plot(np.arange(0, len(data['time'])), lst1,color='purple',linewidth=0.75,label='LFC') #plotting LFC line
        plt.plot(np.arange(0, len(data['time'])), lst2,color='blue',linewidth=0.75,label='LCL') #plotting LCL line

        #plotting all times the parcel is near the CL including where it doesn't go above LFC
        whereCL=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air_directory/whereCL_4_0622.nc')['maxconv_x'] #***
        print('finding all points of CL')
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            y=parcel['y'].isel(time=t,xh=xh).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
            z=parcel['z'].isel(time=t,xh=xh).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1; 
            if z<750/1000:
                maxconv_x=whereCL.isel(time=t,y=which_y,z=which_z);
                tf=np.any(np.isin(maxconv_x,np.arange(which_x-2,which_x+3)))
                if tf==True:
                    plt.scatter(t,z,color='red') #plotting additional max CL points

    plt.xlabel('x (km)')
    plt.ylabel('z (km)')
    plt.legend();plt.title(f"Parcel {out_nz[ind,0]} Trajectory")           
    plt.savefig(os.path.join(folder_path, f"parcel_{out_nz[ind,0]} >{zthresh} km.png")) 
    plt.close()    
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds") #5 minutes for 20 parcels #12-14 minutes for 100 parcels 
#25 minutes for 250 parcels
#25 minutes for 450 parcels
#1 hour for 2000 parcels